# 자녀 가치관 기반 유저 매칭 추천 시스템

이 노트북은 설문조사 데이터를 기반으로 선호동물상 외관과 자녀 가치관을 분석하여  
**유사도 기반 추천**과 **보완도 기반 추천**을 제공하는 매칭 시스템을 구현합니다.

---
## 목차
1. 데이터 불러오기
2. 데이터 전처리
3. 피처 엔지니어링 (One-hot, 중요도 가중치, MBTI형 성향 점수)
4. 유사도 기반 매칭 추천
5. 보완도 기반 매칭 추천
6. 통합 추천 시스템
---

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터 불러오기

In [2]:
# CSV 파일 경로 설정 (필요시 수정)
FILE_PATH = "./data_hj/저출산_소개팅_설문조사_시나리오추가_더미확장_10000rows_나의동물상추가.csv"

df = pd.read_csv(FILE_PATH)
print(f"데이터 shape: {df.shape}")
df.head()

데이터 shape: (10000, 25)


,타임스탬프,0. 당신의 성함,0. 당신의 이상형,0_1. 나의 동물상,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",...,맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도",Unnamed: 23
0,2026/02/27 3:53:18 PM GMT+9,강현준,꽃돼지상,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,...,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3,https://drive.google.com/u/0/open?usp=forms_we...
1,2026/02/27 4:15:15 PM GMT+9,임경빈,꼬부기상,사자상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,...,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3,NaN
2,2026/02/27 4:19:32 PM GMT+9,이정미,곰상,너구리상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,...,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4,NaN
3,2026/02/27 4:29:59 PM GMT+9,쳌,고양이상,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,...,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2,NaN
4,2026/02/27 4:30:18 PM GMT+9,김용희,여우상,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,...,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 25 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   타임스탬프                                                                                  10000 non-null  str  
 1   0. 당신의 성함                                                                              10000 non-null  str  
 2   0. 당신의 이상형                                                                             10000 non-null  str  
 3   0_1. 나의 동물상                                                                            10000 non-null  str  
 4   1-1. 희망하는 자녀 수                                                                         10000 non-null  str  
 5   1-2. 희망하는 자녀 구성                                                                        10000 non-nul

In [4]:
# 원본 데이터 백업
df_raw = df.copy()

# 2. 데이터 전처리

## (1) 컬럼명 정리

In [5]:
def clean_colname(col: str) -> str:
    """컬럼명 정규화: 공백/따옴표/줄바꿈 정리"""
    col = str(col).strip()
    col = col.replace("\n", " ").replace("\t", " ")
    col = re.sub(r"\s+", " ", col)
    col = col.replace('"', "")
    return col

# 정규화 적용
cleaned_cols = [clean_colname(c) for c in df.columns]

# 중복 컬럼명 처리
seen = {}
unique_cols = []
for c in cleaned_cols:
    if c in seen:
        seen[c] += 1
        unique_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        unique_cols.append(c)

df.columns = unique_cols
print("정리된 컬럼 목록:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

정리된 컬럼 목록:
  0: 타임스탬프
  1: 0. 당신의 성함
  2: 0. 당신의 이상형
  3: 0_1. 나의 동물상
  4: 1-1. 희망하는 자녀 수
  5: 1-2. 희망하는 자녀 구성
  6: 1-3. 자녀 갖고 싶은 시기
  7: 1-4. 생물학적 출산이 어려움 발생 시 대안
  8: 1. 자녀 계획 및 가족 구성 항목에 대해 중요도
  9: 아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?
  10: 평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.
  11: 경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?
  12: 재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?
  13: 두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?
  14: 한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.
  15: 맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?
  16: 양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?
  17: 주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?
  18: 아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?
  19: 4-1. 자녀 교육비/양육비 지출 비중
  20: 4-2. 육아 휴직, 양육 부담
  21: 4. 경제적 지원 및 가사 분담에 대해 중요도
  22: 5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?
  23: 5. 자녀 가치관에 대해 중요도
  24: Unnamed: 23


## (2) 컬럼 리네임

In [6]:
# 컬럼 리네임 매핑
RENAME_MAP = {
    '0. 당신의 성함': 'user_name',
    '0. 당신의 이상형': 'ideal_type',
    '0_1. 나의 동물상' : 'my_type',
    '1-1. 희망하는 자녀 수': 'p_children_count',
    '1-2. 희망하는 자녀 구성': 'p_children_composition',
    '1-3. 자녀 갖고 싶은 시기': 'p_children_timing',
    '1-4. 생물학적 출산이 어려움 발생 시 대안': 'p_infertility_alternative',
    '1. 자녀 계획 및 가족 구성 항목에 대해 중요도': 'imp_family_plan',
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?': 'sc_toothbrushing',
    '평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.': 'sc_bedtime_story',
    '경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?': 'sc_competition_2nd',
    '재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?': 'sc_talent_education',
    '두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?': 'sc_discipline_conflict',
    '한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.': 'sc_play_vs_chores',
    '맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?': 'sc_grandparents_help',
    '양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?': 'sc_inlaws_advice',
    '주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?': 'sc_rainy_zoo',
    '아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?': 'sc_education_fund_risk',
    '4-1. 자녀 교육비/양육비 지출 비중': 'e_childcare_cost_share',
    '4-2. 육아 휴직, 양육 부담': 'e_parental_leave_burden',
    '4. 경제적 지원 및 가사 분담에 대해 중요도': 'imp_econ_housework',
    '5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?': 'child_values_open',
    '5. 자녀 가치관에 대해 중요도': 'imp_child_values'
}

# 정확한 매칭이 안 되는 경우를 위한 부분 매칭
def find_and_rename(df, rename_map):
    new_rename = {}
    for old_name, new_name in rename_map.items():
        # 정확한 매칭 시도
        if old_name in df.columns:
            new_rename[old_name] = new_name
        else:
            # 부분 매칭 시도 (핵심 키워드로)
            for col in df.columns:
                # 핵심 키워드 추출하여 매칭
                key_parts = old_name.split()[:3]  # 앞 3단어
                if all(part in col for part in key_parts[:2]):
                    new_rename[col] = new_name
                    break
    return new_rename

actual_rename = find_and_rename(df, RENAME_MAP)
df = df.rename(columns=actual_rename)

print(f"리네임된 컬럼 수: {len(actual_rename)}")
print("\n현재 컬럼 목록:")
print(list(df.columns))

리네임된 컬럼 수: 23

현재 컬럼 목록:
['타임스탬프', 'user_name', 'ideal_type', 'my_type', 'p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative', 'imp_family_plan', 'sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk', 'e_childcare_cost_share', 'e_parental_leave_burden', 'imp_econ_housework', 'child_values_open', 'imp_child_values', 'Unnamed: 23']


## (3) 불필요한 컬럼 제거 및 인덱스 설정

In [7]:
# 타임스탬프 및 불필요한 컬럼 제거
drop_cols = ['타임스탬프']
# Unnamed 컬럼도 제거
drop_cols += [c for c in df.columns if 'Unnamed' in c]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

# user_name을 인덱스로 설정하기 전에 중복 확인
if 'user_name' in df.columns:
    # 중복된 user_name이 있으면 번호 붙이기
    if df['user_name'].duplicated().any():
        print("⚠️ 중복된 user_name 발견, 고유하게 변환합니다.")
        name_counts = {}
        unique_names = []
        for name in df['user_name']:
            if name in name_counts:
                name_counts[name] += 1
                unique_names.append(f"{name}_{name_counts[name]}")
            else:
                name_counts[name] = 0
                unique_names.append(name)
        df['user_name'] = unique_names
    
    df = df.set_index('user_name')

print(f"최종 데이터 shape: {df.shape}")
print(f"인덱스 고유값 수: {df.index.nunique()} / 전체: {len(df.index)}")
df.head()

⚠️ 중복된 user_name 발견, 고유하게 변환합니다.
최종 데이터 shape: (10000, 22)
인덱스 고유값 수: 10000 / 전체: 10000


,ideal_type,my_type,p_children_count,p_children_composition,p_children_timing,p_infertility_alternative,imp_family_plan,sc_toothbrushing,sc_bedtime_story,sc_competition_2nd,...,sc_play_vs_chores,sc_grandparents_help,sc_inlaws_advice,sc_rainy_zoo,sc_education_fund_risk,e_childcare_cost_share,e_parental_leave_burden,imp_econ_housework,child_values_open,imp_child_values
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,꽃돼지상,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,5,...,4,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3
임경빈,꼬부기상,사자상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,4,...,4,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3
이정미,곰상,너구리상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,5,...,5,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4
쳌,고양이상,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,5,...,5,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2
김용희,여우상,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,4,...,1,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3


## (4) 컬럼 그룹 정의

In [8]:
# 컬럼 그룹 정의
META_COLS = ['ideal_type', 'my_type']

#자녀계획 및 가족 구성
PLAN_COLS = ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']

#시나리오 
SCENARIO_COLS = ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education',
                 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 
                 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']

#경제관, 양육관
ECON_COLS = ['e_childcare_cost_share', 'e_parental_leave_burden']

#자녀 가치관
VAL_COLS = ['child_values_open']

IMPORTANCE_COLS = ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']

# 존재하는 컬럼만 필터링
all_groups = {
    'META_COLS': [c for c in META_COLS if c in df.columns],
    'PLAN_COLS': [c for c in PLAN_COLS if c in df.columns],
    'SCENARIO_COLS': [c for c in SCENARIO_COLS if c in df.columns],
    'ECON_COLS': [c for c in ECON_COLS if c in df.columns],
    'VAL_COLS': [c for c in VAL_COLS if c in df.columns],
    'IMPORTANCE_COLS': [c for c in IMPORTANCE_COLS if c in df.columns]
}

for gname, cols in all_groups.items():
    print(f"{gname}: {len(cols)}개 - {cols}")

META_COLS: 2개 - ['ideal_type', 'my_type']
PLAN_COLS: 4개 - ['p_children_count', 'p_children_composition', 'p_children_timing', 'p_infertility_alternative']
SCENARIO_COLS: 10개 - ['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']
ECON_COLS: 2개 - ['e_childcare_cost_share', 'e_parental_leave_burden']
VAL_COLS: 1개 - ['child_values_open']
IMPORTANCE_COLS: 3개 - ['imp_family_plan', 'imp_econ_housework', 'imp_child_values']


# 3. 피처 엔지니어링

## (1) 범주형 변수 One-Hot Encoding (중요도 가중치 적용)

In [9]:
def one_hot_with_importance(df, col, importance_col=None, known_categories=None):
    """
    범주형 컬럼을 One-Hot 인코딩하고, 중요도 가중치를 적용합니다.
    복수 응답(세미콜론 구분)도 처리합니다.
    
    ★ 수정: iloc 기반 인덱싱으로 중복 인덱스 문제 해결
    """
    if col not in df.columns:
        return pd.DataFrame(index=df.index)
    
    # 복수 응답 분리 및 카테고리 수집
    all_categories = set()
    for val in df[col].dropna():
        for item in str(val).split(';'):
            item = item.strip()
            if item:
                all_categories.add(item)
    
    # 알려진 카테고리가 있으면 그 외는 '기타'로
    if known_categories:
        final_categories = list(known_categories) + ['기타']
    else:
        final_categories = list(all_categories)
    
    # One-Hot 인코딩 결과 초기화
    result = pd.DataFrame(0.0, index=df.index, columns=[f"{col}_{cat}" for cat in final_categories])
    
    # 중요도 가중치 (1~5점을 0.2~1.0으로 변환)
    if importance_col and importance_col in df.columns:
        weights = df[importance_col].fillna(3).astype(float) / 5.0
    else:
        weights = pd.Series(1.0, index=df.index)
    
    # ★ 수정: iloc 기반으로 행 순회 (중복 인덱스 문제 해결)
    for i in range(len(df)):
        idx = df.index[i]  # 실제 인덱스 값
        val = df.iloc[i][col]  # iloc으로 값 가져오기
        
        # 스칼라 값인지 확인 후 결측 체크
        if pd.isna(val):
            continue
        
        val_str = str(val)
        items = [item.strip() for item in val_str.split(';') if item.strip()]
        n_items = len(items)
        
        if n_items == 0:
            continue
        
        # 가중치도 iloc으로 가져오기
        weight = float(weights.iloc[i])
        
        for item in items:
            if known_categories and item not in known_categories:
                col_name = f"{col}_기타"
            else:
                col_name = f"{col}_{item}"
            
            if col_name in result.columns:
                # ★ iloc으로 값 설정
                result.iloc[i, result.columns.get_loc(col_name)] = weight / n_items
    
    return result

In [10]:
# 알려진 카테고리 정의
KNOWN_CATEGORIES = {
    'p_children_count': ['1명', '2명', '3명', '그 이상'],
    'p_children_composition': ['오직 딸', '오직 아들', '딸 1명, 아들 1명'],
    'p_children_timing': ['결혼 즉시', '결혼 후 1~2년 이내', '결혼 후 3~5년 이내', '경제적 안정 후'],
    'p_infertility_alternative': ['의학적 도움 적극 시도', '입양 고려', '무자녀'],
    'e_childcare_cost_share': ['노후보단 자녀 교육', '노후 먼저, 남는 예산으로 지원'],
    'e_parental_leave_burden': ['경제력 높은 사람 일하고, 한명은 전담 육아', '맞벌이하면서 외부 도움(조부모, 시터)'],
    'child_values_open': ['경제적 성공, 사회적 지위', '도덕적, 타인 배려', '자신이 좋아하는 일, 행복', '회복탄력성, 생활력 강한 사람']
}

# 중요도 매핑
IMPORTANCE_MAPPING = {
    'p_children_count': 'imp_family_plan',
    'p_children_composition': 'imp_family_plan',
    'p_children_timing': 'imp_family_plan',
    'p_infertility_alternative': 'imp_family_plan',
    'e_childcare_cost_share': 'imp_econ_housework',
    'e_parental_leave_burden': 'imp_econ_housework',
    'child_values_open': 'imp_child_values'
}

# One-Hot Encoding 수행
onehot_dfs = []
for col, categories in KNOWN_CATEGORIES.items():
    if col in df.columns:
        imp_col = IMPORTANCE_MAPPING.get(col)
        oh_df = one_hot_with_importance(df, col, imp_col, categories)
        onehot_dfs.append(oh_df)
        print(f"{col}: {oh_df.shape[1]}개 피처 생성")

# 모든 One-Hot 컬럼 합치기
if onehot_dfs:
    df_onehot = pd.concat(onehot_dfs, axis=1)
else:
    df_onehot = pd.DataFrame(index=df.index)
    
print(f"\n총 One-Hot 피처 수: {df_onehot.shape[1]}")
df_onehot.head()

p_children_count: 5개 피처 생성
p_children_composition: 4개 피처 생성
p_children_timing: 5개 피처 생성
p_infertility_alternative: 4개 피처 생성
e_childcare_cost_share: 3개 피처 생성
e_parental_leave_burden: 3개 피처 생성
child_values_open: 5개 피처 생성

총 One-Hot 피처 수: 29


,p_children_count_1명,p_children_count_2명,p_children_count_3명,p_children_count_그 이상,p_children_count_기타,p_children_composition_오직 딸,p_children_composition_오직 아들,"p_children_composition_딸 1명, 아들 1명",p_children_composition_기타,p_children_timing_결혼 즉시,...,"e_childcare_cost_share_노후 먼저, 남는 예산으로 지원",e_childcare_cost_share_기타,"e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아","e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)",e_parental_leave_burden_기타,"child_values_open_경제적 성공, 사회적 지위","child_values_open_도덕적, 타인 배려","child_values_open_자신이 좋아하는 일, 행복","child_values_open_회복탄력성, 생활력 강한 사람",child_values_open_기타
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.2,0.0,0.2,0.0,0.0,0.0,0.0,0.6,0.0,0.0
임경빈,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,...,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.6,0.0
이정미,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,...,0.0,0.0,0.0,0.6,0.0,0.0,0.0,0.0,0.8,0.0
쳌,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,...,0.0,0.0,0.4,0.0,0.0,0.0,0.0,0.4,0.0,0.0
김용희,0.0,0.4,0.0,0.0,0.0,0.0,0.0,0.4,0.0,0.0,...,0.0,0.0,0.0,0.4,0.0,0.0,0.0,0.6,0.0,0.0


In [21]:
df_onehot.to_csv('./data_hj/데이터_자녀가치관_전처리.csv')

## (4) 시나리오 응답 기반 MBTI형 성향 점수 계산

5개 축:
- **S-F** (Strict-Flexible): 양육 실행 스타일
- **A-H** (Achievement-Happiness): 교육/성장 우선순위  
- **E-L** (Equal-Lead): 공동양육 운영 방식
- **B-T** (Boundary-Together): 확장가족/경계
- **P-G** (Plan-Go): 계획/리스크 대응

In [11]:
# 시나리오 컬럼이 존재하는지 확인
scenario_cols = [c for c in SCENARIO_COLS if c in df.columns]
print(f"찾은 시나리오 컬럼 수: {len(scenario_cols)}")
print(scenario_cols)

찾은 시나리오 컬럼 수: 10
['sc_toothbrushing', 'sc_bedtime_story', 'sc_competition_2nd', 'sc_talent_education', 'sc_discipline_conflict', 'sc_play_vs_chores', 'sc_grandparents_help', 'sc_inlaws_advice', 'sc_rainy_zoo', 'sc_education_fund_risk']


In [12]:
# 시나리오 문항을 2개씩 묶어서 5개 축으로
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

# 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름)
AXES = [
    ("S", "F", "parenting_style_SF"),      # 양육 실행 스타일
    ("A", "H", "education_priority_AH"),   # 교육/성장 우선순위
    ("E", "L", "co_parenting_mode_EL"),    # 공동양육 운영 방식
    ("B", "T", "family_boundary_BT"),      # 확장가족/경계
    ("P", "G", "planning_risk_PG"),        # 계획/리스크 대응
]

print(f"축 개수: {len(AXES)}, 페어 개수: {len(pairs)}")
for i, (pair, axis) in enumerate(zip(pairs, AXES)):
    print(f"  축 {i+1} ({axis[2]}): {pair[0]} + {pair[1]}")

축 개수: 5, 페어 개수: 5
  축 1 (parenting_style_SF): sc_toothbrushing + sc_bedtime_story
  축 2 (education_priority_AH): sc_competition_2nd + sc_talent_education
  축 3 (co_parenting_mode_EL): sc_discipline_conflict + sc_play_vs_chores
  축 4 (family_boundary_BT): sc_grandparents_help + sc_inlaws_advice
  축 5 (planning_risk_PG): sc_rainy_zoo + sc_education_fund_risk


In [13]:
def calculate_trait_scores(df, pairs, axes):
    """
    시나리오 응답으로부터 성향 점수 계산
    응답: 1-5점 (1=왼쪽 성향, 5=오른쪽 성향, 3=중립)
    """
    trait_scores = {}
    mbti_letters = {}
    
    for i, ((col1, col2), (left, right, axis_name)) in enumerate(zip(pairs, axes)):
        if col1 not in df.columns or col2 not in df.columns:
            continue
        
        # 응답값 합산 (결측값은 3으로 대체)
        v1 = pd.to_numeric(df[col1], errors='coerce').fillna(3).astype(float)
        v2 = pd.to_numeric(df[col2], errors='coerce').fillna(3).astype(float)
        
        # 두 문항 합산 (2~10점 범위)
        total = v1 + v2
        
        # 0~1 범위로 정규화
        trait_scores[axis_name] = (total - 2) / 8.0
        
        # MBTI 스타일 문자 결정 (각 값을 개별적으로 판단)
        def get_letter(x):
            if x < 6:
                return left
            elif x > 6:
                return right
            else:
                return left  # 중립은 왼쪽으로
        
        mbti_letters[axis_name + '_letter'] = total.apply(get_letter)
    
    return pd.DataFrame(trait_scores), pd.DataFrame(mbti_letters)

df_traits, df_mbti = calculate_trait_scores(df, pairs, AXES)
print(f"성향 점수 shape: {df_traits.shape}")
df_traits.head()
# df_mbti.head()

성향 점수 shape: (10000, 5)


,parenting_style_SF,education_priority_AH,co_parenting_mode_EL,family_boundary_BT,planning_risk_PG
user_name,,,,,
강현준,0.500,0.875,0.750,0.500,0.875
임경빈,0.625,0.500,0.750,0.750,0.625
이정미,0.875,0.875,0.875,0.500,0.500
쳌,0.625,0.875,0.750,0.500,0.625
김용희,0.375,0.500,0.375,0.625,0.375


In [14]:
df_traits.to_csv('./data_hj/데이터_MBTI숫자_전처리.csv')

In [15]:
# MBTI 유형 조합 생성
def create_mbti_type(row):
    letters = []
    for col in row.index:
        val = row[col]
        if pd.notna(val):
            letters.append(str(val))
    return ''.join(letters)

df_mbti['childcare_mbti'] = df_mbti.apply(create_mbti_type, axis=1)
print("자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts())

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    1010
SHLBP     776
SHEBG     752
SHEBP     714
SHLTG     555
FHLTG     503
FHEBP     481
SHETG     479
FHLBG     463
FHLBP     461
FHEBG     419
SHLTP     371
SHETP     361
SALBP     345
FHETG     299
FHLTP     294
SALTP     260
SALBG     218
SAEBG     168
FHETP     158
SAEBP     139
SAETP     102
FALTG     101
FALBP      88
SALTG      81
SAETG      75
FAEBP      70
FAEBG      66
FALTP      64
FALBG      55
FAETG      52
FAETP      20
Name: count, dtype: int64


In [24]:
df_mbti.to_csv('./data_hj/데이터_MBTI유형_전처리.csv')

In [16]:
df_mbti.head(1)

,parenting_style_SF_letter,education_priority_AH_letter,co_parenting_mode_EL_letter,family_boundary_BT_letter,planning_risk_PG_letter,childcare_mbti
user_name,,,,,,
강현준,S,H,L,B,G,SHLBG


## (5) 자녀가치관, MBTI 숫자 피처 합치기

In [17]:
# 모든 피처 합치기
df_features = pd.concat([df_onehot, df_traits], axis=1)

# 결측값 처리
df_features = df_features.fillna(0)

print(f"최종 피처 매트릭스 shape: {df_features.shape}")
print(f"\n피처 목록:")
for i, col in enumerate(df_features.columns):
    print(f"  {i+1}. {col}")

최종 피처 매트릭스 shape: (10000, 34)

피처 목록:
  1. p_children_count_1명
  2. p_children_count_2명
  3. p_children_count_3명
  4. p_children_count_그 이상
  5. p_children_count_기타
  6. p_children_composition_오직 딸
  7. p_children_composition_오직 아들
  8. p_children_composition_딸 1명, 아들 1명
  9. p_children_composition_기타
  10. p_children_timing_결혼 즉시
  11. p_children_timing_결혼 후 1~2년 이내
  12. p_children_timing_결혼 후 3~5년 이내
  13. p_children_timing_경제적 안정 후
  14. p_children_timing_기타
  15. p_infertility_alternative_의학적 도움 적극 시도
  16. p_infertility_alternative_입양 고려
  17. p_infertility_alternative_무자녀
  18. p_infertility_alternative_기타
  19. e_childcare_cost_share_노후보단 자녀 교육
  20. e_childcare_cost_share_노후 먼저, 남는 예산으로 지원
  21. e_childcare_cost_share_기타
  22. e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아
  23. e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)
  24. e_parental_leave_burden_기타
  25. child_values_open_경제적 성공, 사회적 지위
  26. child_values_open_도덕적, 타인 배려
  27. child_values_open_자신이 좋아하는 일, 행복
  28. c

In [18]:
# 피처 그룹 정의 (추천 시스템에서 사용)
FEATURE_GROUPS = {
    'value_cols': [c for c in df_features.columns if c.startswith(('p_', 'e_', 'child_'))],
    'trait_cols': [c for c in df_features.columns if c in ['parenting_style_SF', 'education_priority_AH', 
                                                           'co_parenting_mode_EL', 'family_boundary_BT', 
                                                           'planning_risk_PG']]
}

print(f"가치관 피처 수: {len(FEATURE_GROUPS['value_cols'])}")
print(f"성향 피처 수: {len(FEATURE_GROUPS['trait_cols'])}")

가치관 피처 수: 29
성향 피처 수: 5


# 4. 추천 알고리즘



![alt text](추천알고리즘3차.png)

## (1) 자녀 가치관 유사도 분석

In [19]:
# 가치관 유사도 매트릭스 계산
value_cols = FEATURE_GROUPS['value_cols']
if len(value_cols) > 0:
    val_sim_matrix = cosine_similarity(df_features[value_cols])
    val_sim_df = pd.DataFrame(val_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"가치관 유사도 매트릭스 shape: {val_sim_df.shape}")
else:
    val_sim_df = pd.DataFrame(1.0, index=df_features.index, columns=df_features.index)

val_sim_df.head()

가치관 유사도 매트릭스 shape: (10000, 10000)


user_name,강현준,임경빈,이정미,쳌,김용희,김진우,김경표,임태나,이나원,김종우,...,서림경_1,전준린,남주지,백담라,유훈나,정수현,허호하,허림수,남별훈,유태호_1
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,1.000000,0.582160,0.356235,0.516388,0.596354,0.152333,0.342748,0.318952,0.328358,0.247540,...,0.304159,0.628372,0.099227,0.554774,0.157572,0.071970,0.369451,0.328358,0.276103,0.392668
임경빈,0.582160,1.000000,0.534258,0.666667,0.696311,0.167984,0.167984,0.281379,0.061633,0.335968,...,0.298142,0.671937,0.262613,0.407850,0.278019,0.317460,0.462738,0.061633,0.314970,0.288675
이정미,0.356235,0.534258,1.000000,0.353553,0.492366,0.160357,0.267261,0.562786,0.313786,0.160357,...,0.221359,0.374166,0.445669,0.486664,0.294884,0.242437,0.473736,0.313786,0.240535,0.489898
쳌,0.516388,0.666667,0.353553,1.000000,0.783349,0.566947,0.283473,0.361773,0.312019,0.377964,...,0.167705,0.850420,0.295439,0.344124,0.573415,0.285714,0.226339,0.312019,0.212605,0.613435
김용희,0.596354,0.696311,0.492366,0.783349,1.000000,0.328976,0.460566,0.503812,0.531085,0.592157,...,0.311400,0.723747,0.411435,0.519170,0.326679,0.397892,0.409767,0.531085,0.164488,0.753778


In [50]:
val_sim_df.to_csv('./data_hj/테이블_자녀가치관유사도.csv')

## (2) MBTI 유사도 분석

In [20]:
# 성향 유사도 매트릭스 계산
trait_cols = FEATURE_GROUPS['trait_cols']
if len(trait_cols) > 0:
    trait_sim_matrix = cosine_similarity(df_features[trait_cols])
    trait_sim_df = pd.DataFrame(trait_sim_matrix, index=df_features.index, columns=df_features.index)
    print(f"성향 유사도 매트릭스 shape: {trait_sim_df.shape}")
else:
    trait_sim_df = pd.DataFrame(1.0, index=df_features.index, columns=df_features.index)

trait_sim_df.head()

성향 유사도 매트릭스 shape: (10000, 10000)


user_name,강현준,임경빈,이정미,쳌,김용희,김진우,김경표,임태나,이나원,김종우,...,서림경_1,전준린,남주지,백담라,유훈나,정수현,허호하,허림수,남별훈,유태호_1
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,1.000000,0.944806,0.945599,0.985331,0.931809,0.970907,0.947168,0.972698,0.968540,0.967973,...,0.943737,0.960125,0.944635,0.974061,0.960357,0.948636,0.941033,0.962920,0.944154,0.945599
임경빈,0.944806,1.000000,0.948026,0.955985,0.970362,0.917417,0.849946,0.865928,0.968257,0.900222,...,0.818031,0.930746,0.921904,0.915702,0.939238,0.963364,0.950181,0.976394,0.886662,0.948026
이정미,0.945599,0.948026,1.000000,0.985372,0.924526,0.978139,0.829205,0.918464,0.932706,0.855843,...,0.894163,0.930568,0.940523,0.964824,0.971951,0.974442,0.956562,0.953539,0.971732,1.000000
쳌,0.985331,0.955985,0.985372,1.000000,0.947389,0.986666,0.902817,0.953642,0.961587,0.925885,...,0.925663,0.967716,0.948475,0.982390,0.981273,0.979903,0.963174,0.971517,0.971378,0.985372
김용희,0.931809,0.970362,0.924526,0.947389,1.000000,0.886844,0.840841,0.828965,0.964209,0.893281,...,0.784825,0.977501,0.850532,0.913139,0.917495,0.966158,0.921851,0.979339,0.857443,0.924526


In [ ]:
trait_sim_df.to_csv('./data_hj/테이블_MBTI유사도.csv')

## (3) 자녀 가치관 유사도와 MBTI 유사도 종합 점수로 정렬

In [21]:
def get_similarity_recommendations(user_name):
    """
    유사도 기반 추천: 가치관과 성향이 비슷한 사람 추천
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    recommendations = []
    
    # 현재 사용자의 위치 (iloc용)
    user_iloc = df_features.index.get_loc(user_name)
    
    for i, other_user in enumerate(df_features.index):
        if user_name == other_user:
            continue
        
        # iloc으로 유사도 점수 가져오기 (스칼라 보장)
        val_score = float(val_sim_df.iloc[user_iloc, i])
            
        
        # MBTI 유형 가져오기 (iloc 사용)
        other_mbti = 'N/A'
        if 'childcare_mbti' in df_mbti.columns and i < len(df_mbti):
            mbti_val = df_mbti.iloc[i]['childcare_mbti']
            other_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'

        #이상형 동물상 가져오기
        other_ideal = 'N/A'
        if 'ideal_type' in df.columns and i < len(df):
            ideal_val = df.iloc[i]['ideal_type']
            other_ideal = str(ideal_val) if pd.notna(ideal_val) else 'N/A'

        #나의 동물상 가져오기
        other_my = 'N/A'
        if 'my_type' in df.columns and i < len(df):
            my_val = df.iloc[i]['my_type']
            other_my = str(my_val) if pd.notna(my_val) else 'N/A'

        recommendations.append({
            'name': other_user,
            'value_similarity': round(val_score, 4),
            'childcare_mbti': other_mbti,
            'ideal_type' : other_ideal,
            'my_type' : other_my

        })
    
    result_df = pd.DataFrame(recommendations)
    result_df = result_df.sort_values('value_similarity', ascending=False)
    result_df = result_df.reset_index(drop=True)
    result_df.index = result_df.index + 1  # 1부터 시작
    
    return result_df

# 테스트
test_user = df_features.index[0]
print(f"\n=== {test_user}님의 유사도 기반 추천 ===")
df_value = get_similarity_recommendations(test_user)
df_value


=== 강현준님의 유사도 기반 추천 ===


,name,value_similarity,childcare_mbti,ideal_type,my_type
1,양영서,1.0000,FHEBG,고양이상,고슴도치상
2,윤민빈,1.0000,SHLBG,여우상,여우상
3,장호아,0.9971,SHEBG,여우상,여우상
4,손진라,0.9957,SHLBP,여우상,고슴도치상
5,윤윤별,0.9951,SHEBP,토끼상,토끼상
...,...,...,...,...,...
9995,김말이,0.0000,SHLBP,여우상,여우상
9996,이솔담,0.0000,FHEBP,강아지상,강아지상
9997,노영나,0.0000,FHLBP,여우상,여우상
9998,하현태,0.0000,FHEBP,곰상,곰상


In [62]:
df_value.to_csv('./data_hj/예시_유사도결과.csv')

In [30]:
df_value.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 1 to 9999
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              9999 non-null   str    
 1   value_similarity  9999 non-null   float64
 2   childcare_mbti    9999 non-null   str    
dtypes: float64(1), str(2)
memory usage: 375.8 KB


## (4) 정렬된 목록에 MBTI 유사형, 보완형 파생 변수 생성

- 유사형 : MBTI 4자리 이상 같은 유형
- 보완형 : MBTI 2자리 이하 같은 유형

In [22]:
def add_mbti_similarity_column(df, target_mbti):
    """
    childcare_mbti 열을 target_mbti와 비교하여 
    'similarity_label' 열(high/low/none)을 추가합니다.
    """
    
    # 1. 일치하는 알파벳 개수 계산 함수
    def count_matches(mbti):
        if pd.isna(mbti) or len(mbti) != 5:
            return -1
        return sum(1 for a, b in zip(mbti, target_mbti) if a == b)

    # 2. 모든 행에 대해 일치 개수 계산
    match_counts = df['childcare_mbti'].apply(count_matches)

    # 3. 조건 설정 (4개 이상은 'high', 2개 이하는 'low')
    conditions = [
        (match_counts >= 4),
        (match_counts <= 2) & (match_counts >= 0)
    ]
    choices = ['유사형', '보완형']

    # 4. 새로운 열 추가 (조건에 해당하지 않는 3개 일치는 'middle' 혹은 None으로 표시)
    df['similarity_label'] = np.select(conditions, choices, default='보통')
    
    # 만약 완전히 Boolean(True/False) 형태로만 보고 싶다면 아래처럼 별도 생성도 가능합니다.
    # df['is_high'] = match_counts >= 4
    # df['is_low'] = match_counts <= 2

    return df

# --- 사용 예시 ---
target_mbti = df_mbti.loc[test_user, 'childcare_mbti']  # 비교 기준이 되는 값
df_value = add_mbti_similarity_column(df_value, target_mbti)

df_value.head()

,name,value_similarity,childcare_mbti,ideal_type,my_type,similarity_label
1,양영서,1.0000,FHEBG,고양이상,고슴도치상,보통
2,윤민빈,1.0000,SHLBG,여우상,여우상,유사형
3,장호아,0.9971,SHEBG,여우상,여우상,유사형
4,손진라,0.9957,SHLBP,여우상,고슴도치상,유사형
5,윤윤별,0.9951,SHEBP,토끼상,토끼상,보통


In [ ]:
# def split_by_mbti(df, target_mbti):
#     """
#     유사도 정렬된 그룹을 사용자의 mbti와 비교하여 분할
#     """
#     # 1. 일치하는 알파벳 개수를 계산하는 내부 함수
#     def count_matches(mbti):
#         # zip을 사용해 같은 위치의 알파벳끼리 비교합니다.
#         # 예: 'FHEBG'와 'SHLBG' 비교 -> [False, True, False, True, True] -> sum은 3
#         return sum(1 for a, b in zip(mbti, target_mbti) if a == b)

#     # 2. 모든 행에 대해 일치 개수 계산 (임시 시리즈 생성)
#     match_series = df['childcare_mbti'].apply(count_matches)

#     # 3. 조건에 맞는 데이터프레임 생성
#     # 4개 이상 일치 (유사함)
#     df_high_similarity = df[match_series >= 4].copy()
    
#     # 2개 이하 일치 (유사하지 않음)
#     df_low_similarity = df[match_series <= 2].copy()

#     return df_high_similarity, df_low_similarity

# # --- 사용 예시 ---
# target_mbti = df_mbti.loc[test_user, 'childcare_mbti']  # 비교 기준이 되는 값
# print(target_mbti)
# df_high, df_low = split_by_mbti(df_value, target_mbti)

# # 파일로 저장하고 싶다면:
# df_high.to_csv('./data_hj/예시_유사도_MBTI유사형.csv', index=False)
# df_low.to_csv('./data_hj/예시_유사도_MBTI보완형.csv', index=False)

SHLBG


## (5) 동물상과 이상형 매칭 여부 파생 변수 추가

- 1번 그룹 : 외모 매칭 성공, 가치관 유사함, MBTI 유사함
- 2번 그룹 : 외모 매칭 실패, 가치관 유사함, MBTI 유사함
- 3번 그룹 : 외모 매칭 성공, 가치관 유사함, MBTI 보완형
- 4번 그룹 : 외모 매칭 실패, 가치관 유사함, MBTI 보완형

In [23]:
target_my = df.loc[test_user, 'my_type']  # 비교 기준이 되는 값
target_ideal = df.loc[test_user, 'ideal_type']
print(f'나의 동물상 : {target_my}, 나의 이상형 : {target_ideal}입니다.')

df_value['is_match'] = (target_my == df_value['ideal_type']) & (target_ideal == df_value['my_type'])

df_value


나의 동물상 : 꽃돼지상, 나의 이상형 : 꽃돼지상입니다.


,name,value_similarity,childcare_mbti,ideal_type,my_type,similarity_label,is_match
1,양영서,1.0000,FHEBG,고양이상,고슴도치상,보통,False
2,윤민빈,1.0000,SHLBG,여우상,여우상,유사형,False
3,장호아,0.9971,SHEBG,여우상,여우상,유사형,False
4,손진라,0.9957,SHLBP,여우상,고슴도치상,유사형,False
5,윤윤별,0.9951,SHEBP,토끼상,토끼상,보통,False
...,...,...,...,...,...,...,...
9995,김말이,0.0000,SHLBP,여우상,여우상,유사형,False
9996,이솔담,0.0000,FHEBP,강아지상,강아지상,보완형,False
9997,노영나,0.0000,FHLBP,여우상,여우상,보통,False
9998,하현태,0.0000,FHEBP,곰상,곰상,보완형,False


In [24]:
df_value.to_csv('./data_hj/예시_유사도_mbti분류_매칭분류.csv')

In [78]:
df_value['is_match'].unique()


array([False,  True])

# 5. 최종 매칭 결과 

### (1) 이상형 매칭, 가치관 유사, MBTI 유사형

In [ ]:

#이상형 매칭 결과, mbti 유사형, 가치관 유사도 가장 높은 한사람 찾기
result_match = df_value[(df_value['is_match'] == True)]

#이상형 매칭 성공 시 
if not result_match.empty:
    #MBTI 유사형 1명 찾기
    result_similar = result_match[result_match['similarity_label'] == '유사형'].head(1)
    
    if not result_similar.empty:
        print(f'추천 매칭은 {result_similar['name'].item()}입니다.')
        print(f'나의 동물상 : {target_my}, 나의 이상형 : {target_ideal}입니다.')
        print(f'{result_similar['name'].item()}님의 동물상 : {result_similar['my_type'].item()}, {result_similar['name'].item()}의 이상형 : {result_similar['ideal_type'].item()}')
        print(f'나의 양육 스타일 : {target_mbti} 입니다.')
        print(f'{result_similar['name'].item()}님의 양육 스타일 : {result_similar['childcare_mbti'].item()} 입니다.')
    else:
        print('이상형 매칭 성공, MBTI 유사형 없음')
    
    result_







추천 매칭은 유혜연입니다.
나의 동물상 : 꽃돼지상, 나의 이상형 : 꽃돼지상입니다.
유혜연님의 동물상 : 꽃돼지상, 유혜연의 이상형 : 꽃돼지상
나의 양육 스타일 : SHLBG 입니다.
유혜연님의 양육 스타일 : SHLBG 입니다.


In [ ]:
# def get_complementary_recommendations(user_name, top_n=5, value_threshold=0.5):
#     """
#     보완도 기반 추천: 가치관은 유사하면서 성향은 보완적인 사람 추천
#     """
#     if user_name not in df_features.index:
#         return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
#     trait_cols = FEATURE_GROUPS['trait_cols']
#     recommendations = []
    
#     # 현재 사용자의 위치
#     user_iloc = df_features.index.get_loc(user_name)
    
#     # 내 MBTI 가져오기
#     my_mbti = 'N/A'
#     if 'childcare_mbti' in df_mbti.columns and user_iloc < len(df_mbti):
#         mbti_val = df_mbti.iloc[user_iloc]['childcare_mbti']
#         my_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'
    
#     for i, other_user in enumerate(df_features.index):
#         if user_name == other_user:
#             continue
        
#         # 가치관 유사도 (iloc 사용)
#         val_score = float(val_sim_df.iloc[user_iloc, i])
        
#         # 가치관 유사도가 기준 이하면 스킵
#         if val_score < value_threshold:
#             continue
        
#         # 보완도 계산
#         comp_score, trait_details = calculate_complementary_score(
#             user_iloc, i, df_features, trait_cols
#         )
        
#         # 종합 점수: 가치관 유사도 * 보완도
#         total_score = val_score * comp_score
        
#         # MBTI 유형 가져오기
#         other_mbti = 'N/A'
#         if 'childcare_mbti' in df_mbti.columns and i < len(df_mbti):
#             mbti_val = df_mbti.iloc[i]['childcare_mbti']
#             other_mbti = str(mbti_val) if pd.notna(mbti_val) else 'N/A'
        
#         # 이상형 매칭 여부
#         ideal_match = False
#         if 'ideal_type' in df.columns:
#             my_ideal = df.iloc[user_iloc]['ideal_type'] if user_iloc < len(df) else None
#             other_ideal = df.iloc[i]['ideal_type'] if i < len(df) else None
#             if pd.notna(my_ideal) and pd.notna(other_ideal):
#                 ideal_match = bool(str(my_ideal) == str(other_ideal))
        
#         recommendations.append({
#             'name': other_user,
#             'value_similarity': round(val_score, 4),
#             'complementary_score': round(comp_score, 4),
#             'total_score': round(total_score, 4),
#             'my_mbti': my_mbti,
#             'their_mbti': other_mbti,
#             'ideal_type_match': ideal_match,
#             'match_type': '보완형(Complementary)'
#         })
    
#     if not recommendations:
#         return f"가치관 유사도 {value_threshold} 이상인 매칭 대상이 없습니다."
    
#     result_df = pd.DataFrame(recommendations)
#     result_df = result_df.sort_values('total_score', ascending=False).head(top_n)
#     result_df = result_df.reset_index(drop=True)
#     result_df.index = result_df.index + 1
    
#     return result_df

# # 테스트
# print(f"\n=== {test_user}님의 보완도 기반 추천 ===")
# get_complementary_recommendations(test_user, top_n=5, value_threshold=0.3)


=== 윤서나님의 보완도 기반 추천 ===


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,고수라,0.8377,0.9369,0.7849,FAETP,SHEBP,False,보완형(Complementary)
2,허담혜,0.9057,0.8355,0.7568,FAETP,SHLTP,False,보완형(Complementary)
3,전아수,0.8377,0.8979,0.7522,FAETP,SHLBP,False,보완형(Complementary)
4,허도하,0.7842,0.9547,0.7486,FAETP,SHEBP,False,보완형(Complementary)
5,남경담,0.8721,0.8321,0.7256,FAETP,SHETG,False,보완형(Complementary)


# 6. 통합 추천 시스템

유사도 기반과 보완도 기반 추천을 모두 제공하는 통합 함수

In [25]:
def get_smart_recommendations(user_name, top_n=5, mode='both'):
    """
    스마트 추천 시스템: 유사도 + 보완도 기반 통합 추천
    
    Parameters:
    -----------
    user_name : str - 추천 대상 사용자
    top_n : int - 각 유형별 추천할 인원 수
    mode : str - 'similar', 'complementary', 'both'
    
    Returns:
    --------
    dict with recommendation DataFrames
    """
    if user_name not in df_features.index:
        return f"사용자 '{user_name}'을(를) 찾을 수 없습니다."
    
    results = {}
    user_iloc = df_features.index.get_loc(user_name)
    
    # 사용자 정보 출력
    print("="*60)
    print(f"🎯 {user_name}님을 위한 맞춤 추천")
    print("="*60)
    
    # 사용자 프로필
    if 'childcare_mbti' in df_mbti.columns and user_iloc < len(df_mbti):
        user_mbti = df_mbti.iloc[user_iloc]['childcare_mbti']
        print(f"\n📋 당신의 자녀양육 MBTI: {user_mbti}")
    
    if 'ideal_type' in df.columns and user_iloc < len(df):
        user_ideal = df.iloc[user_iloc]['ideal_type']
        print(f"💕 당신의 이상형: {user_ideal}")
    
    # 유사도 기반 추천
    if mode in ['similar', 'both']:
        print(f"\n\n🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람")
        print("-"*50)
        similar_recs = get_similarity_recommendations(user_name, top_n)
        results['similar'] = similar_recs
        if isinstance(similar_recs, pd.DataFrame):
            display(similar_recs)
        else:
            print(similar_recs)
    
    # 보완도 기반 추천
    if mode in ['complementary', 'both']:
        print(f"\n\n🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람")
        print("-"*50)
        comp_recs = get_complementary_recommendations(user_name, top_n, value_threshold=0.3)
        results['complementary'] = comp_recs
        if isinstance(comp_recs, pd.DataFrame):
            display(comp_recs)
        else:
            print(comp_recs)
    
    print("\n" + "="*60)
    
    return results

In [26]:
# 통합 추천 테스트
test_user = df_features.index[0]
results = get_smart_recommendations(test_user, top_n=5, mode='both')

🎯 윤서나님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FAETP
💕 당신의 이상형: 오리상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,남지현,0.9634,0.8391,0.9261,SHLTG,False,유사형(Similar)
2,오은우,0.9035,0.8343,0.8828,SHLTP,False,유사형(Similar)
3,한지솔,0.9035,0.8300,0.8814,SHLTP,False,유사형(Similar)
4,이예성_1,0.8771,0.8871,0.8801,FHEBP,False,유사형(Similar)
5,백찬수,0.8733,0.8814,0.8757,FHLBP,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,고수라,0.8377,0.9369,0.7849,FAETP,SHEBP,False,보완형(Complementary)
2,허담혜,0.9057,0.8355,0.7568,FAETP,SHLTP,False,보완형(Complementary)
3,전아수,0.8377,0.8979,0.7522,FAETP,SHLBP,False,보완형(Complementary)
4,허도하,0.7842,0.9547,0.7486,FAETP,SHEBP,False,보완형(Complementary)
5,남경담,0.8721,0.8321,0.7256,FAETP,SHETG,False,보완형(Complementary)


In [27]:
# 여러 사용자 테스트
print("\n" + "#"*70)
print("# 다른 사용자 추천 결과")
print("#"*70)

for user in list(df_features.index)[1:4]:  # 2~4번째 사용자
    print("\n")
    get_smart_recommendations(user, top_n=3, mode='both')


######################################################################
# 다른 사용자 추천 결과
######################################################################


🎯 권은준님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FALTP
💕 당신의 이상형: 고양이상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,강혜호,0.9398,0.9690,0.9485,FALBG,False,유사형(Similar)
2,강영서,0.9234,0.9743,0.9387,FHLBG,False,유사형(Similar)
3,배윤은,0.9139,0.9892,0.9365,FALTG,True,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,남경혜,0.8658,0.8143,0.7050,FALTP,SHETG,False,보완형(Complementary)
2,조솔림_1,0.8346,0.8108,0.6767,FALTP,SHETG,False,보완형(Complementary)
3,안은찬,0.8193,0.8108,0.6643,FALTP,SHEBG,True,보완형(Complementary)





🎯 강린지님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: SHETP
💕 당신의 이상형: 강아지상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


...

,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,한호주,0.9359,0.9191,0.8603,SHETP,FHLBP,False,보완형(Complementary)
2,남민나_1,0.9641,0.8143,0.7850,SHETP,FALBG,False,보완형(Complementary)
3,고연도,0.9641,0.7395,0.7129,SHETP,FHLBP,False,보완형(Complementary)





🎯 강태성님을 위한 맞춤 추천

📋 당신의 자녀양육 MBTI: FHLTG
💕 당신의 이상형: 곰상


🤝 [유사도 기반 추천] - 나와 비슷한 가치관을 가진 사람
--------------------------------------------------


,name,value_similarity,trait_similarity,total_score,childcare_mbti,ideal_type_match,match_type
1,한준라,1.0,0.9978,0.9994,SHLTG,False,유사형(Similar)
2,전연라,1.0,0.9920,0.9976,FHLTG,False,유사형(Similar)
3,허태태,1.0,0.9667,0.9900,SHLTG,False,유사형(Similar)




🔄 [보완도 기반 추천] - 가치관은 같고 성향은 보완적인 사람
--------------------------------------------------


,name,value_similarity,complementary_score,total_score,my_mbti,their_mbti,ideal_type_match,match_type
1,신하하_1,0.8165,0.8979,0.7331,FHLTG,FHEBP,True,보완형(Complementary)
2,전현별_1,1.0000,0.6915,0.6915,FHLTG,FHLTG,False,보완형(Complementary)
3,남겸겸,0.8347,0.8108,0.6768,FHLTG,SHLBP,False,보완형(Complementary)


## 상세 분석: 두 사용자 간 매칭 상세 정보

In [28]:
def analyze_match_detail(user1, user2):
    """
    두 사용자 간의 매칭 상세 분석
    """
    if user1 not in df_features.index or user2 not in df_features.index:
        return "사용자를 찾을 수 없습니다."
    
    user1_iloc = df_features.index.get_loc(user1)
    user2_iloc = df_features.index.get_loc(user2)
    
    print("="*60)
    print(f"📊 {user1} ↔ {user2} 매칭 상세 분석")
    print("="*60)
    
    # 1. 가치관 유사도
    val_score = float(val_sim_df.iloc[user1_iloc, user2_iloc])
    print(f"\n🎯 가치관 유사도: {val_score:.2%}")
    
    # 2. 성향 비교
    trait_cols = FEATURE_GROUPS['trait_cols']
    if trait_cols:
        print(f"\n📈 양육 성향 비교:")
        print("-"*50)
        print(f"{'축':<25} {user1:<10} {user2:<10} {'차이':>10}")
        print("-"*50)
        
        for col in trait_cols:
            col_idx = df_features.columns.get_loc(col)
            v1 = float(df_features.iloc[user1_iloc, col_idx])
            v2 = float(df_features.iloc[user2_iloc, col_idx])
            diff = abs(v1 - v2)
            print(f"{col:<25} {v1:.3f}      {v2:.3f}      {diff:.3f}")
    
    # 3. MBTI 비교
    if 'childcare_mbti' in df_mbti.columns:
        mbti1 = str(df_mbti.iloc[user1_iloc]['childcare_mbti'])
        mbti2 = str(df_mbti.iloc[user2_iloc]['childcare_mbti'])
        print(f"\n🧬 자녀양육 MBTI:")
        print(f"   {user1}: {mbti1}")
        print(f"   {user2}: {mbti2}")
        
        # 같은 문자 개수
        same_count = sum(1 for a, b in zip(mbti1, mbti2) if a == b)
        print(f"   일치하는 축: {same_count}/{min(len(mbti1), len(mbti2))}")
    
    # 4. 이상형 비교
    if 'ideal_type' in df.columns:
        ideal1 = str(df.iloc[user1_iloc]['ideal_type'])
        ideal2 = str(df.iloc[user2_iloc]['ideal_type'])
        print(f"\n💕 이상형:")
        print(f"   {user1}: {ideal1}")
        print(f"   {user2}: {ideal2}")
    
    # 5. 종합 평가
    comp_score, _ = calculate_complementary_score(user1_iloc, user2_iloc, df_features, trait_cols)
    print(f"\n🏆 종합 평가:")
    print(f"   유사도 점수: {val_score:.2%}")
    print(f"   보완도 점수: {comp_score:.2%}")
    
    if val_score >= 0.7 and comp_score >= 0.5:
        print(f"   → ⭐⭐⭐ 최고의 매칭! (가치관 일치 + 적절한 보완)")
    elif val_score >= 0.5:
        print(f"   → ⭐⭐ 좋은 매칭 (가치관 유사)")
    elif comp_score >= 0.5:
        print(f"   → ⭐⭐ 보완적 매칭 (성향 보완)")
    else:
        print(f"   → ⭐ 보통 매칭")
    
    print("="*60)

# 테스트
users = list(df_features.index[:2])
if len(users) >= 2:
    analyze_match_detail(users[0], users[1])

📊 윤서나 ↔ 권은준 매칭 상세 분석

🎯 가치관 유사도: 50.78%

📈 양육 성향 비교:
--------------------------------------------------
축                         윤서나        권은준                차이
--------------------------------------------------
parenting_style_SF        0.875      0.625      0.250
education_priority_AH     0.500      0.500      0.000
co_parenting_mode_EL      0.125      0.750      0.625
family_boundary_BT        0.750      0.750      0.000
planning_risk_PG          0.500      0.500      0.000

🧬 자녀양육 MBTI:
   윤서나: FAETP
   권은준: FALTP
   일치하는 축: 4/5

💕 이상형:
   윤서나: 오리상
   권은준: 고양이상

🏆 종합 평가:
   유사도 점수: 50.78%
   보완도 점수: 40.14%
   → ⭐⭐ 좋은 매칭 (가치관 유사)


# 7. 데이터 저장 및 요약

In [29]:
# 처리된 피처 데이터 저장
output_features_path = "processed_features.csv"
df_features.to_csv(output_features_path)
print(f"피처 데이터 저장: {output_features_path}")

# MBTI 데이터 저장
output_mbti_path = "childcare_mbti.csv"
df_mbti.to_csv(output_mbti_path)
print(f"MBTI 데이터 저장: {output_mbti_path}")

# 유사도 매트릭스 저장
output_sim_path = "similarity_matrix.csv"
val_sim_df.to_csv(output_sim_path)
print(f"유사도 매트릭스 저장: {output_sim_path}")

피처 데이터 저장: processed_features.csv
MBTI 데이터 저장: childcare_mbti.csv
유사도 매트릭스 저장: similarity_matrix.csv


In [30]:
print("\n" + "="*70)
print("📊 최종 요약")
print("="*70)
print(f"\n총 사용자 수: {len(df_features)}명")
print(f"피처 수: {df_features.shape[1]}개")
print(f"  - 가치관 피처: {len(FEATURE_GROUPS['value_cols'])}개")
print(f"  - 성향 피처: {len(FEATURE_GROUPS['trait_cols'])}개")

print(f"\n자녀양육 MBTI 유형 분포:")
print(df_mbti['childcare_mbti'].value_counts().head(10))

print(f"\n이상형 분포:")
if 'ideal_type' in df.columns:
    print(df['ideal_type'].value_counts())

print("\n" + "="*70)
print("✅ 추천 시스템 구축 완료!")
print("="*70)


📊 최종 요약

총 사용자 수: 9966명
피처 수: 34개
  - 가치관 피처: 29개
  - 성향 피처: 5개

자녀양육 MBTI 유형 분포:
childcare_mbti
SHLBG    1006
SHLBP     773
SHEBG     749
SHEBP     712
SHLTG     554
FHLTG     501
FHEBP     478
SHETG     477
FHLBG     462
FHLBP     460
Name: count, dtype: int64

이상형 분포:
ideal_type
여우상      2383
곰상       1621
고양이상     1587
강아지상     1314
토끼상       550
오리상       512
꽃돼지상      256
꼬부기상      248
쿼카상       247
너구리상      237
호랑이상      110
햄스터상      107
다람쥐상      105
공룡상       105
판다상       102
사슴상        99
사자상        99
고슴도치상      98
늑대상        95
펭귄상        91
Name: count, dtype: int64

✅ 추천 시스템 구축 완료!
